# Start here: paper coverage and reproducible validation

**Question.** Which paper claims can a reader reproduce here, which outputs are certificates, and which claims remain outside this finite validation?

**Paper locations.** Section V (HGP), Section VI and Appendix G.1–G.2 (LP/LCS), Section VIII and Appendix G.4–G.5 (BP), Section IX (bicycle/BB), Appendix E (nonabelian examples), and Appendix F, Propositions 1–2 (construction correspondences).

**Companion paper.** Yuan Yao, Guo Zhang, Yingli Yang, Yuanye Zhu, Xiaosi Xu, Xiao Yuan, and Ying Li, **“An introduction to qLDPC codes from a structural perspective.”** Submitted.

The paper numbers refer to the manuscript identified by `PAPER_SOURCE` in
[`paper_cases.py`](../qldpc_generator/paper_cases.py). Source labels are retained
to help locate the same material after renumbering. Equation numbers below are
local to this notebook; code outputs are computational results.

**Before running.** Follow the [installation instructions](../README.md#installation-and-execution),
select that Python environment, and run the cells in order. All matrix arithmetic
and ranks are over GF(2); matrix indices start at zero. The reading routes below link to the individual mathematical walkthroughs.

## 1. Choose a reading route

The assumed background is binary linear algebra and the CSS stabilizer
formalism. Orbit and group-algebra conventions are introduced in the relevant
walkthroughs. No knowledge of the library's internal classes is required.

| Question | Walkthrough | Paper location | Evidence displayed |
| --- | --- | --- | --- |
| Why is the HGP candidate logical, and why is D = 3? | [01 — HGP](01-hgp-and-distance.ipynb) | Sec. V, Eq. (44) and distance argument | Complete small matrices, quotient image, stabilizer failure control, exact distance |
| What is lifted, and what do the LP/LCS examples establish? | [02 — LP/LCS](02-lp-lcs.ipynb) | Definition 4; Apps. D, G.1, G.2 | Explicit polynomials, formal adjoint, shared-coordinate size, ranks and LCS certificate |
| How do boundaries descend to orbit classes? | [03 — BP](03-example-bp.ipynb) | Sec. VIII; Apps. G.4–G.5, Algorithm 2 | Orbit trace, XOR and source-counting controls, folded matrices |
| Do the LP–BP and FB–BP maps respect both boundaries? | [04 — Correspondences](04-fb-and-correspondences.ipynb) | App. F, Propositions 1–2 | Maps on entire orbits, shared qubit permutations, residuals and rejected hypotheses |
| What determines K for BB, and which examples are different? | [05 — Bicycle/BB](05-bicycle-bb.ipynb) | Sec. IX, Eqs. (90), (93)–(94) | Quotient monomials, ranks, direct sums and optional coordinate/boundary examples |

For the two main proof checks, read **01 and 04** after this overview. For a
construction-by-construction route, read **01 → 02 → 03 → 04 → 05**.
The notebooks contain saved outputs, and each can also be run independently
from its first cell. Equations are numbered locally within each notebook.

In [1]:
import numpy as np
from IPython.display import Markdown, display

def show_table(headers, rows):
    """Format results for reading; no mathematical calculations happen here."""
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines += ["| " + " | ".join(map(str, row)) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

## 2. Regenerate the finite-instance report

The default run checks 12 specified inputs and performs bounded distance
searches where configured. It usually completes quickly for these inputs;
raising the search limits can be expensive. Every exact distance requires a
logical witness and exclusion of smaller weights. A lower bound alone is
printed with an inequality.

Set `RUN_DISTANCE_SEARCH = False` for algebra and correspondence checks only.
In that mode the distance column must say **not searched**, even when an old
saved report contained a certificate. Results below are generated by this run.

In [2]:
from collections import Counter
from time import perf_counter
from qldpc_generator.verify_paper import verify_all

RUN_DISTANCE_SEARCH = True
started = perf_counter()
report = verify_all(distance=RUN_DISTANCE_SEARCH)
assert report['passed']
print(f"All executed checks passed in {perf_counter() - started:.2f}s.")
print("Environment:", {key: report['provenance'][key] for key in ('python', 'numpy', 'sympy')})

locations = {
    'hgp13': 'Sec. V', 'lp544': 'App. G.1', 'lp714': 'App. G.1', 'lp1020': 'App. G.1',
    'lcs52': 'App. G.2', 'bp90': 'App. G.5', 'bp108': 'App. G.5',
    'bicycle6': 'Sec. IX', 'gb48': 'Sec. IX', 'bb72': 'Sec. IX, BB family',
    'bb144': 'Sec. IX, Eqs. (93)–(94)', 'mitten150': 'App. E, Eq. (E22)',
}

def claim_text(claim):
    if claim['value'] is None:
        return 'not specified'
    symbol = '<=' if claim['kind'] == 'reported_upper_bound' else '='
    return f"D {symbol} {claim['value']}"

def computation_text(result):
    if result['status'] == 'exact':
        return f"D = {result['value']}"
    if result['status'] == 'no_logicals':
        return 'K = 0; no nontrivial logical distance'
    if result['status'] == 'not_searched':
        return 'not searched'
    lo, hi = result.get('lower_bound'), result.get('upper_bound')
    if lo is not None and hi is not None:
        return f"{lo} <= D <= {hi}"
    if lo is not None:
        return f"D >= {lo}"
    return f"D <= {hi}" if hi is not None else 'no bound established'

rows = []
for item in report['cases']:
    p = item['parameters']
    result = item['distance_computation']
    rows.append([item['id'], locations[item['id']], p['n'], p['k'],
                 claim_text(item['distance_claim']), computation_text(result), result['status']])
show_table(["Instance", "Paper location", "N", "K", "Paper distance claim", "Local distance result", "Status"], rows)
counts = Counter(item['distance_computation']['status'] for item in report['cases'])
print("Distance statuses:", dict(counts))
if RUN_DISTANCE_SEARCH:
    assert counts == {'exact': 5, 'bounded': 4, 'not_searched': 3}
else:
    assert counts == {'not_searched': 12}

All executed checks passed in 2.64s.
Environment: {'python': '3.14.3', 'numpy': '2.5.4', 'sympy': '1.14.0'}


| Instance | Paper location | N | K | Paper distance claim | Local distance result | Status |
| --- | --- | --- | --- | --- | --- | --- |
| hgp13 | Sec. V | 13 | 1 | D = 3 | D = 3 | exact |
| lp544 | App. G.1 | 544 | 80 | D <= 12 | not searched | not_searched |
| lp714 | App. G.1 | 714 | 100 | D <= 16 | not searched | not_searched |
| lp1020 | App. G.1 | 1020 | 136 | D <= 20 | not searched | not_searched |
| lcs52 | App. G.2 | 52 | 4 | D = 4 | D = 4 | exact |
| bp90 | App. G.5 | 90 | 8 | D = 10 | D >= 7 | bounded |
| bp108 | App. G.5 | 108 | 8 | D = 8 | D >= 7 | bounded |
| bicycle6 | Sec. IX | 6 | 2 | D = 2 | D = 2 | exact |
| gb48 | Sec. IX | 48 | 6 | D = 8 | D = 8 | exact |
| bb72 | Sec. IX, BB family | 72 | 12 | D = 6 | D = 6 | exact |
| bb144 | Sec. IX, Eqs. (93)–(94) | 144 | 12 | D = 12 | D >= 7 | bounded |
| mitten150 | App. E, Eq. (E22) | 150 | 30 | D = 10 | D >= 7 | bounded |

Distance statuses: {'exact': 5, 'not_searched': 3, 'bounded': 4}


### Read the claims and results separately

For BP90, a literature value D = 10 and a local result D >= 7 are compatible,
but the latter does not reproduce the former. Likewise, the three large LP
distance upper bounds remain literature claims because no local witnesses
for them are provided by this run. The general LCS distance formula is still
a conjecture, even though the displayed LCS52 instance has exact distance four.

`passed` means the **executed** algebra, correspondence, and certificate checks
passed and no checked result contradicted its recorded claim. It does not mean
that every paper statement has been proved or every reported distance verified.

The report stores the X/Z logical supports for certified distances. The next
cell shows one such certificate and independently rechecks its syndrome and
nonmembership in the stabilizer row space through the CSS interface.

In [3]:
from qldpc_generator.paper_cases import paper_cases
case = next(case for case in paper_cases() if case.identifier == 'bb72')
entry = next(item for item in report['cases'] if item['id'] == case.identifier)
distance = entry['distance_computation']
if distance['status'] == 'exact':
    instance = case.build()
    rows = []
    for sector, data in distance['sectors'].items():
        vector = np.zeros(instance.n, dtype=np.uint8)
        vector[data['witness']] = 1
        assert instance.is_logical(vector, sector) and int(vector.sum()) == data['value']
        rows.append([sector, data['value'], data['witness'], True])
    show_table(["BB72 sector", "Weight", "Support (zero-based)", "Logical witness rechecked"], rows)
else:
    print("Distance search disabled: no certificate is claimed in this run.")

| BB72 sector | Weight | Support (zero-based) | Logical witness rechecked |
| --- | --- | --- | --- |
| X | 6 | [0, 3, 6, 15, 24, 30] | True |
| Z | 6 | [0, 1, 2, 3, 19, 23] | True |

## 3. Match relation checks to mathematical statements

Each name below stands for a finite calculation, not a proof of a whole
family. In particular, the correspondence checks compare both complete
matrices under a common qubit permutation. Notebook 04 exposes the orbit
maps and demonstrates why the nonabelian multiplication order matters.

In [4]:
relation_descriptions = {
    'lp_bp_C2': ('Proposition 1', 'LP–BP and converse on C2; notebook 04 explains the map'),
    'lp_bp_C4': ('Proposition 1', 'LP–BP and converse on C4'),
    'lp_bp_S3': ('Proposition 1', 'Nonabelian LP–BP and converse on S3'),
    'fb_bp_S3_nonfree_fiber': ('Proposition 2', 'FB–BP and covering converse with nonfree fiber'),
    'bb_direct_sum_t3': ('Sec. IX', 'Complete permutation for three BB72 components'),
    'bb144_quotient_dimension': ('Eq. (90)', 'Binary rank agrees with polynomial quotient dimension'),
    'bb144_generalized_toric': ('Sec. IX', 'Monomial translations of BB144'),
    'hgp_distance_quotient_witness': ('Sec. V', 'Quotient certificate for the HGP upper bound'),
    'classical_path_star_soundness': ('Eq. (102); App. G.6', 'Finite normalized classical soundness'),
}
show_table(["Paper location", "Executed relation check", "Passed"], [
    [*relation_descriptions[key], passed] for key, passed in report['relations'].items()
])

| Paper location | Executed relation check | Passed |
| --- | --- | --- |
| Proposition 1 | LP–BP and converse on C2; notebook 04 explains the map | True |
| Proposition 1 | LP–BP and converse on C4 | True |
| Proposition 1 | Nonabelian LP–BP and converse on S3 | True |
| Proposition 2 | FB–BP and covering converse with nonfree fiber | True |
| Sec. IX | Complete permutation for three BB72 components | True |
| Eq. (90) | Binary rank agrees with polynomial quotient dimension | True |
| Sec. IX | Monomial translations of BB144 | True |
| Sec. V | Quotient certificate for the HGP upper bound | True |
| Eq. (102); App. G.6 | Finite normalized classical soundness | True |

## 4. State what is still missing

Parameters alone do not specify a code. These inputs are absent from the
manuscript used by this implementation, so the report does not claim to
reproduce their exact check matrices. No substitution by a similarly sized
periodic or surface code is made.

In [5]:
show_table(["Unresolved instance", "Missing input"], [
    [item['claim'], item['reason']] for item in report['unresolved_inputs']
])
display(Markdown("**Outside finite matrix validation:**\n\n" + "\n".join(
    f"- {item}" for item in report['outside_finite_validation']
)))

| Unresolved instance | Missing input |
| --- | --- |
| GB [[126,28,8]] | Paper mentions parameters but does not specify its two polynomials. |
| Planar [[88,6,6]] and tile [[288,8,12]], [[512,18,19]] | Full open-boundary check data are not supplied in this manuscript; periodic templates do not determine them. |

**Outside finite matrix validation:**

- General asymptotic rate/distance theorems
- General LCS distance conjecture
- Decoder thresholds and single-shot guarantees
- Hardware experimental results

## 5. Optional: why the same classical code can have different soundness

Section X, Eq. (102), and Appendix G.6 (`eq:ltc_soundness`, `app:testability-details`)
discuss the dependence on the check presentation. The finite classical
quantity used here is

$$
\rho(H)=\min_{v\notin\ker H}
\frac{|Hv|/m}{d_H(v,\ker H)/n},\qquad H\in\mathbb F_2^{m\times n}. \tag{1}
$$

Both path checks and star checks define the length-eight repetition code.
They have seven rows and classical distance eight. The star has stronger
normalized soundness, but its central bit participates in seven checks,
compared with maximum degree two for the path. The denominator is distance
to the classical kernel, not distance modulo quantum stabilizers.

In [6]:
from fractions import Fraction
from qldpc_generator.representations import repetition_checks, classical_soundness
from qldpc_generator.certificates import classical_distance
from qldpc_generator.gf2 import nullspace

path = repetition_checks(8)
star = repetition_checks(8, 'star')
assert np.array_equal(nullspace(path), nullspace(star))
rows = []
for name, checks, expected in [('path', path, Fraction(2, 7)), ('star', star, Fraction(8, 7))]:
    rho = classical_soundness(checks)
    d = classical_distance(checks)['value']
    assert rho == expected and d == 8
    rows.append([name, checks.shape, d, str(rho), int(checks.sum(0).max())])
show_table(["Presentation", "Check shape", "Classical distance", "Normalized soundness", "Maximum bit degree"], rows)

| Presentation | Check shape | Classical distance | Normalized soundness | Maximum bit degree |
| --- | --- | --- | --- | --- |
| path | (7, 8) | 8 | 2/7 | 2 |
| star | (7, 8) | 8 | 8/7 | 7 |

This is a finite classical comparison. It is not a claim that the star is an
LDPC family, nor a verification of quantum local testability or single-shot
error correction.

## 6. Reproduce and inspect the evidence

From the repository root, after installing the dependencies:

```bash
python -m pytest -q
python -m qldpc_generator.verify_paper --output reports/paper_validation.json
python scripts/execute_notebooks.py
```

The JSON report contains software versions, source-code hashes, paper identity,
per-instance ranks, distance status, and available witnesses. The notebook
runner uses the invoking Python environment, starts a fresh kernel for every
file, and saves the outputs. A cell failure stops the run.

Inspect [paper inputs](../qldpc_generator/paper_cases.py),
[report generation](../qldpc_generator/verify_paper.py),
[independent distance-oracle tests](../tests/test_algebra_css.py), and
[correspondence and failure tests](../tests/test_products.py).
The [README](../README.md) explains conventions and execution limits.